In [1]:
import pandas as pd

In [2]:
hourly_station = pd.read_csv("../data/raw/tspr2025_skytrainavgalightsbrdgs_yearstationdaytypehourly.csv")

station_year = pd.read_csv("../data/raw/tspr2025_skytrain_yearstation.csv")

rolling_hour = pd.read_csv("../data/raw/tspr2025_rail_rollinghouravgpassengervol.csv")

busiest_segments = pd.read_csv("../data/raw/tspr2025_rail_busiestsegavgvol_yearlinedaytypesegtimeperiod.csv")

daily_segments = pd.read_csv("../data/raw/tspr2025_rail_avgdailypassengervol_yearsegentdaytype.csv")

line_ridership = pd.read_csv("../data/raw/tspr2025_rail_ridershipbrdgs_yearline.csv")

In [3]:
print("Datasets loaded successfully")

Datasets loaded successfully


In [4]:
hourly_station_clean = hourly_station.copy()
station_year_clean = station_year.copy()
rolling_hour_clean = rolling_hour.copy()
busiest_segments_clean = busiest_segments.copy()
daily_segments_clean = daily_segments.copy()
line_ridership_clean = line_ridership.copy()

In [5]:
import re

def to_snake_case(column):
    column = re.sub(r'(?<!^)(?=[A-Z])', '_', column)
    column = column.replace("__", "_")
    return column.lower()

clean_datasets = [
    hourly_station_clean,
    station_year_clean,
    rolling_hour_clean,
    busiest_segments_clean,
    daily_segments_clean,
    line_ridership_clean
]

for df in clean_datasets:
    df.columns = [to_snake_case(col) for col in df.columns]

In [6]:
hourly_station_clean.columns.tolist()

['calendar_year',
 'station_name',
 'day_type',
 'hour',
 'average_daily_station_alightings',
 'average_daily_station_boardings']

In [9]:
for df in clean_datasets:
    text_columns = df.select_dtypes(include="str").columns

    for col in text_columns:
        df[col] = df[col].str.strip()

In [8]:
hourly_station_clean.head()

,calendar_year,station_name,day_type,hour,average_daily_station_alightings,average_daily_station_boardings
0,2023,22nd Street Station,Sun/Hol,3:00 to 4:00 PM,393,450
1,2023,22nd Street Station,Sun/Hol,3:00 to 4:00 AM,0,0
2,2023,22nd Street Station,Sun/Hol,2:00 to 3:00 PM,363,460
3,2023,22nd Street Station,Sun/Hol,2:00 to 3:00 AM,0,0
4,2023,22nd Street Station,Sun/Hol,12:00 to 1:00 PM,306,421


In [10]:
day_type_map = {
    "MF": "Weekday",
    "Sat": "Saturday",
    "Sun/Hol": "Sunday/Holiday"
}

for df in [
    hourly_station_clean,
    rolling_hour_clean,
    busiest_segments_clean,
    daily_segments_clean
]:
    df["day_type"] = df["day_type"].replace(day_type_map)

In [11]:
hourly_station_clean["day_type"].unique()

<StringArray>
['Sunday/Holiday', 'Saturday', 'Weekday']
Length: 3, dtype: str

In [12]:
line_ridership_clean = line_ridership_clean[
    line_ridership_clean["mode"] != "West Coast Express"
].copy()

In [13]:
line_ridership_clean["mode"].unique()

<StringArray>
['Canada Line', 'Expo/Millennium Line']
Length: 2, dtype: str

In [15]:
time_parts = hourly_station_clean["hour"].str.extract(
    r'^(?P<time>\d{1,2}:\d{2})(?:\s+(?P<period>AM|PM))?'
)

end_period = hourly_station_clean["hour"].str.extract(
    r'(AM|PM)$'
)[0]

time_parts["period"] = time_parts["period"].fillna(end_period)

hourly_station_clean["hour_24"] = pd.to_datetime(
    time_parts["time"] + " " + time_parts["period"],
    format="%I:%M %p"
).dt.hour

In [16]:
hourly_station_clean[
    ["hour", "hour_24"]
].drop_duplicates().sort_values("hour_24")

,hour,hour_24
5,12:00 to 1:00 AM,0
11,1:00 to 2:00 AM,1
3,2:00 to 3:00 AM,2
1,3:00 to 4:00 AM,3
26,4:00 to 5:00 AM,4
28,5:00 to 6:00 AM,5
30,6:00 to 7:00 AM,6
32,7:00 to 8:00 AM,7
34,8:00 to 9:00 AM,8
13,9:00 to 10:00 AM,9


In [17]:
seg_mode_map = {
    "C": "Canada Line",
    "M": "Expo/Millennium Line"
}

for df in [
    rolling_hour_clean,
    busiest_segments_clean,
    daily_segments_clean
]:
    df["seg_mode"] = df["seg_mode"].replace(seg_mode_map)

In [18]:
rolling_hour_clean["seg_mode"].unique()

<StringArray>
['Canada Line', 'Expo/Millennium Line']
Length: 2, dtype: str

In [ ]:
### Missing station short codes

Missing values in `from_stn_short` and `to_stn_short` correspond to Nanaimo Station. Since the full station name is available and the short code is not required for the analysis, these records were retained without imputation.

In [20]:
print("Day types:", hourly_station_clean["day_type"].unique())
print("Hour range:", sorted(hourly_station_clean["hour_24"].unique()))
print("Segment modes:", rolling_hour_clean["seg_mode"].unique())
print("Line modes:", line_ridership_clean["mode"].unique())

print("\nRows after cleaning:")
print("Hourly station:", len(hourly_station_clean))
print("Station year:", len(station_year_clean))
print("Rolling hour:", len(rolling_hour_clean))
print("Busiest segments:", len(busiest_segments_clean))
print("Daily segments:", len(daily_segments_clean))
print("Line ridership:", len(line_ridership_clean))

Day types: <StringArray>
['Sunday/Holiday', 'Saturday', 'Weekday']
Length: 3, dtype: str
Hour range: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23)]
Segment modes: <StringArray>
['Canada Line', 'Expo/Millennium Line']
Length: 2, dtype: str
Line modes: <StringArray>
['Canada Line', 'Expo/Millennium Line']
Length: 2, dtype: str

Rows after cleaning:
Hourly station: 11413
Station year: 161
Rolling hour: 86792
Busiest segments: 126
Daily segments: 990
Line ridership: 6


In [21]:
hourly_station_clean.to_csv(
    "../data/processed/hourly_station_clean.csv", index=False
)

station_year_clean.to_csv(
    "../data/processed/station_year_clean.csv", index=False
)

rolling_hour_clean.to_csv(
    "../data/processed/rolling_hour_clean.csv", index=False
)

busiest_segments_clean.to_csv(
    "../data/processed/busiest_segments_clean.csv", index=False
)

daily_segments_clean.to_csv(
    "../data/processed/daily_segments_clean.csv", index=False
)

line_ridership_clean.to_csv(
    "../data/processed/line_ridership_clean.csv", index=False
)

print("Cleaned datasets saved successfully")

Cleaned datasets saved successfully
